# P07 · Cuándo conviene dejar Pandas

> **¿Qué se gana y qué se pierde al leer completo, por bloques, con DuckDB o con Polars?**

stop_times.txt del GTFS vigente: 1 098 717 filas y la misma agregación por parada.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/rendimiento_gtfs/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib 'duckdb>=1.0' 'polars>=1.0,<2'
!python scripts/fetch_data.py gtfs-red-movilidad


## 1 · El problema

«Pandas no escala» y «DuckDB es más rápido» son afirmaciones incompletas. La decisión
depende del volumen, la operación, la memoria disponible y el costo de preparar los
datos. P07 fija una sola agregación y exige que todos los motores devuelvan exactamente
lo mismo antes de comparar tiempos.


## 2 · La solución que se va a aplicar

Se cuentan filas de horario por parada con lectura completa, seis bloques, una consulta
DuckDB y un plan lazy de Polars. Se separa el tiempo de extracción del ZIP de cada
consulta, se mide la memoria del DataFrame o bloque y se hace una unión por clave para
encontrar cualquier diferencia. La conclusión se expresa como criterio, no como ganador
universal.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.rendimiento_gtfs import solution as rendimiento

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Fijar una agregación idéntica


Todos cuentan filas por `stop_id`; cambiar la pregunta invalidaría el benchmark.

> ⚠️ Comparar una lectura selectiva con otra que carga ocho columnas mide cosas distintas.


In [ ]:
{'archivo': 'stop_times.txt', 'operacion': 'COUNT(*) GROUP BY stop_id', 'columnas': ['stop_id']}


### Paso 2 · Declarar qué se mide


Tiempo, memoria de datos, bloques y filas se guardan en la misma tabla.

> ⚠️ Un único tiempo total oculta que DuckDB necesita preparar un CSV temporal.


In [ ]:
['segundos','lectura_s','agregacion_s','memoria_datos_mb','bloques','filas_entrada']


### Paso 3 · Ejecutar Pandas completo


Es la referencia simple mientras el archivo cabe en memoria.

> ⚠️ El DataFrame usa más memoria que los 51 MB del CSV porque cada texto es un objeto.


In [ ]:
referencia = rendimiento.cargar_referencia()
referencia['benchmark'].query("metodo == 'Pandas completo'")


### Paso 4 · Ejecutar Pandas por bloques


Cada bloque se agrega y los parciales se suman por clave.

> ⚠️ Una mediana exacta no se puede combinar sumando medianas de bloques.


In [ ]:
referencia['benchmark'].query("metodo == 'Pandas por bloques'")


### Paso 5 · Ejecutar DuckDB y Polars


Ambos consultan el CSV extraído y reportan la extracción aparte.

> ⚠️ DuckDB y Polars lazy no son equivalentes internamente; solo comparten la pregunta y la salida.


In [ ]:
referencia['benchmark'].query("metodo in ['DuckDB sobre CSV', 'Polars lazy sobre CSV']")


### Paso 6 · Exigir igualdad antes de mirar velocidad


Una implementación rápida y distinta es incorrecta.

> ⚠️ Comparar solo el total puede ocultar errores que se compensan entre paradas.


In [ ]:
referencia['validacion']


### Paso 7 · Comparar tiempo y memoria


Ningún método domina las dos dimensiones en todos los volúmenes.

> ⚠️ El color de 'ganador' borra que bloques compra memoria a cambio de tiempo.


In [ ]:
referencia['benchmark']


### Paso 8 · Tomar una decisión proporcional


Se elige el camino más simple que cumple el límite real de recursos.

> ⚠️ Extrapolar linealmente una sola corrida no sustituye medir el siguiente orden de magnitud.


In [ ]:
{'este_feed': 'Pandas completo cabe', 'memoria_restringida': 'bloques', 'muchos_feeds_o_consultas': 'DuckDB/Polars'}


## 4 · El resultado

En este feed Pandas completo cabe y es simple; bloques reduce cerca de 81 % la memoria de datos. DuckDB y Polars ejecutan planes columnares con la preparación informada aparte.

## 5 · Qué queda aprendido

1. Un benchmark empieza por igualdad de resultado.
2. Tiempo y memoria son objetivos distintos.
3. La agregación por bloques exige una operación combinable.
4. Los tiempos se vuelven a medir por equipo.
